# M4L2 - Making Figures with `ggplot2`

Last lab you used R to store values, combine them with operators, and handle data frames. This lab turns those data frames into pictures. Every figure is built with **`ggplot2`**, the plotting package of the tidyverse, and every figure comes from real data: the GSE79275 count table from the Mendoza-Cozatl lab (Khan et al. 2018), which compares the iron-signalling mutant *opt3-2* with wild-type Col-0 in roots and shoots, with and without cadmium.

The *opt3-2* mutant cannot move iron signals properly from the leaves to the roots, so its roots behave as if the plant were starving for iron even when iron is plentiful. If our plots work, that should be easy to see in the iron-uptake genes.

You do not need to understand the statistics of RNA-seq yet. The aim of this lab is to learn how a `ggplot2` figure is put together, one layer at a time, so that the DE plots in M4L3 and the practices make sense.

**Kernel:** use the `Module 3 - R` kernel from M3P2. If that kernel is not available, the course-wide Module 3 environment (see `environment/module_3_rnaseq_salmon.yml`) or the Binder environment (`binder/environment.yml`) has every package this notebook needs.

## Contents
* [Getting the data ready](#setup)
* [The three ingredients of a plot](#ingredients)
* [Comparing two groups gene by gene](#scatter)
* [Looking at distributions](#distributions)
* [Small multiples with facets](#facets)
* [Making a plot readable](#polish)
* [When more is less](#busy)
* [Exercises](#exercises)
* [Looking ahead: a heat map of DE genes](#ahead)

```{tip}
**Where to write R outside this course.** A Jupyter notebook is handy for short lessons, but for longer analyses an editor built for R is more comfortable. [RStudio Desktop](https://posit.co/download/rstudio-desktop/) is free and runs on any laptop ([getting-started guide](https://docs.posit.co/ide/user/ide/get-started/)). On Hellbender you can open RStudio or Code Server from the RSS OnDemand dashboard; we use RStudio on the cluster in Module 6.
```

<a id='setup'></a>
## 1. Getting the data ready

First load the packages. `tidyverse` loads `ggplot2` together with `dplyr`, `tidyr`, `readr` and friends, so one `library()` call is enough. The start-up message that lists the attached packages and any "conflicts" is normal.

In [ ]:
library(tidyverse)

The raw count table is downloaded by `data/fetch_data.sh geo` (Binder does this for you). Each row is one Arabidopsis gene, named by its AGI code (for example `AT4G19690`), and each column is one sequencing library. The cell below reads the table and shows the first rows and the number of rows and columns.

In [ ]:
DATA_DIR <- Sys.getenv("DATA_DIR", unset = "../../../data")  # repository data/ folder; run data/fetch_data.sh geo first
counts <- read_tsv(file.path(DATA_DIR, "geo", "GSE79275", "GSE79275_raw_read_counts.txt"),
                   show_col_types = FALSE)
dim(counts)
counts[1:5, 1:6]

### Decoding the sample names

The library names pack the whole experimental design into a few letters. `ORT2_3.bam`, for instance, reads as:

| Letter(s) | Meaning |
|---|---|
| `O` (or `C`) | genotype: *opt3-2* (or Col-0) |
| `R` (or `S`) | tissue: root (or shoot) |
| `T` (if present) | treated with cadmium; no `T` means the untreated, iron-replete control |
| `2` | biological replicate 2 |
| `_3` | sequencing run (technical replicate) 3 |

`ggplot2` works best with **long** ("tidy") data: one row per observation, one column per variable. Our table is **wide**: one column per library. `pivot_longer()` from `tidyr` stacks the library columns into two new columns, `library` and `count`.

Next we add columns that describe each library. Two functions do the work:

* `grepl(pattern, text)` asks "does this text contain the pattern?" and answers with `TRUE` or `FALSE` for every element. The pattern is a *regular expression*: `^O` means "starts with O", and `^.R` means "any first character, then R".
* `ifelse(test, value_if_true, value_if_false)` turns those `TRUE`/`FALSE` answers into labels.

Inside `mutate()` these build the new columns `genotype`, `tissue` and `treatment`. `treatment` is also turned into a *factor* with the control listed first: `ggplot2` orders categories on axes and in legends by factor level, and alphabetically otherwise. Finally we convert raw counts to **counts per million (CPM)**, which divides each count by the size of its library so that libraries sequenced to different depths can be compared. (DESeq2 uses a better method, which you will meet in M4P1; CPM is fine for drawing pictures.)

In [ ]:
long <- counts %>%
  pivot_longer(cols = -AGI, names_to = "library", values_to = "count") %>%
  mutate(
    library   = sub("\\.bam$", "", library),                     # drop the file extension
    genotype  = ifelse(grepl("^O", library), "opt3-2", "Col-0"),
    tissue    = ifelse(grepl("^.R", library), "root", "shoot"),
    treatment = factor(ifelse(grepl("T[0-9]", library), "Cd", "control"),
                       levels = c("control", "Cd")),                # control first in legends
    bio_rep   = sub("^[A-Z]+([0-9]+)_.*$", "\\1", library),      # the digit(s) before the underscore
    group     = paste(genotype, tissue, treatment)
  ) %>%
  group_by(library) %>%
  mutate(cpm = count / sum(count) * 1e6) %>%                     # counts per million, within each library
  ungroup()

head(long)
nrow(long)   # genes x libraries

That is roughly two million rows. Long tables get big quickly, but each row is now one simple fact: "gene X had this many reads in library Y, which came from this genotype, tissue and treatment".

For several plots we only need a handful of genes that are known to respond to iron status. The vector below gives each AGI code a short name. The `filter()` keeps those rows, and `factor()` fixes the order in which the genes will appear in plots.

In [ ]:
iron_genes <- c(
  IRT1   = "AT4G19690",  # iron transporter in the root epidermis
  FRO2   = "AT1G01580",  # ferric-chelate reductase that frees Fe for IRT1
  FIT    = "AT2G28160",  # transcription factor that switches on IRT1 and FRO2
  bHLH38 = "AT3G56970",  # partner of FIT
  bHLH39 = "AT3G56980",  # partner of FIT
  OPT3   = "AT4G16370",  # the gene mutated in opt3-2
  NAS4   = "AT1G56430",  # nicotianamine synthase (Fe chelator)
  BTS    = "AT3G18290"   # BRUTUS, an E3 ligase that dampens the iron response
)

iron <- long %>%
  filter(AGI %in% iron_genes) %>%
  mutate(gene = factor(names(iron_genes)[match(AGI, iron_genes)], levels = names(iron_genes)))

iron %>% count(gene)

<a id='ingredients'></a>
## 2. The three ingredients of a plot

Every `ggplot2` figure is assembled from the same parts:

1. **Data**: a data frame, passed to `ggplot()`.
2. **Aesthetic mappings**, written inside `aes()`: which column goes to which visual property. `x` and `y` are aesthetics, and so are `colour`, `fill`, `shape`, `size` and `alpha` (transparency).
3. **Geometric objects** ("geoms"): the marks that are drawn, such as points (`geom_point()`), bars (`geom_col()`), boxes (`geom_boxplot()`) or lines (`geom_line()`).

The pieces are joined with `+`. A useful pattern to keep in your head is:

```r
ggplot(<data frame>, aes(x = <column>, y = <column>, ...)) +
  geom_<kind>() +
  <more layers: scales, facets, labels, themes>
```

Let's start with a quality check that people really do in RNA-seq: how many reads does each library have? First we collapse the long table to one row per library.

In [ ]:
lib_sizes <- long %>%
  group_by(library, genotype, tissue, treatment, group) %>%
  summarise(total_reads    = sum(count),
            genes_detected = sum(count > 0),
            .groups = "drop")
head(lib_sizes)

In [ ]:
ggplot(lib_sizes, aes(x = library, y = total_reads)) +
  geom_col()

The plot works, but 72 vertical names are unreadable and nothing tells us which bar belongs to which group. Two small changes help:

* map `fill` to `group`, so the bars are coloured by experimental group (a legend appears automatically);
* add `coord_flip()`, which swaps the axes so the names run horizontally.

Notice that `fill = group` goes **inside** `aes()`, because it depends on a column of the data. Look at how `ggplot2` picks the colours and builds the legend for you.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 10)   # figure size in the notebook (inches)

ggplot(lib_sizes, aes(x = library, y = total_reads, fill = group)) +
  geom_col() +
  coord_flip()

**Inside or outside `aes()`?** Anything that should change *with the data* is a mapping and belongs inside `aes()`. Anything that should be the *same for every mark* is a fixed setting and goes outside, as an argument of the geom. Compare the two plots below: the first colours every point by tissue, the second paints every point dark blue and makes it larger.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)

# colour depends on a column -> inside aes()
ggplot(lib_sizes, aes(x = total_reads, y = genes_detected, colour = tissue)) +
  geom_point()

# colour and size are constants -> outside aes(), in the geom
ggplot(lib_sizes, aes(x = total_reads, y = genes_detected)) +
  geom_point(colour = "darkblue", size = 3)

What would happen if you wrote `aes(colour = "darkblue")`? Try it. `ggplot2` treats `"darkblue"` as a one-level variable, colours the points with its first default colour (a salmon pink) and adds a legend entry called "darkblue". This is one of the most common `ggplot2` mistakes.

**Question:** In the coloured scatter plot, do root and shoot libraries detect a similar number of genes? Does detecting more genes depend on sequencing depth?

<a id='scatter'></a>
## 3. Comparing two groups gene by gene

A classic first look at an RNA-seq experiment is a scatter plot with one point per gene: average expression in the control on the x axis and in the mutant on the y axis. Genes that do not change sit on the diagonal; genes that go up in the mutant sit above it.

We use the untreated roots only. `pivot_wider()` is the reverse of `pivot_longer()`: it gives Col-0 and *opt3-2* their own columns so they can be mapped to `x` and `y`. We work with `log2(CPM + 1)`, because expression spans several orders of magnitude and the `+ 1` avoids taking the log of zero.

In [ ]:
root_means <- long %>%
  filter(tissue == "root", treatment == "control") %>%
  group_by(AGI, genotype) %>%
  summarise(mean_cpm = mean(cpm), .groups = "drop") %>%
  pivot_wider(names_from = genotype, values_from = mean_cpm) %>%
  mutate(log_col0 = log2(`Col-0` + 1),
         log_opt3 = log2(`opt3-2` + 1)) %>%
  filter(`Col-0` > 0 | `opt3-2` > 0)        # drop genes with no reads at all

head(root_means)

In [ ]:
ggplot(root_means, aes(x = log_col0, y = log_opt3)) +
  geom_point()

With more than 20,000 genes most points sit on top of each other, so the dense middle looks like a solid blob. Some fixes, each added as one more layer or argument:

* `alpha = 0.2` makes each point mostly transparent, so dense areas look darker than sparse ones;
* `geom_abline(slope = 1, intercept = 0)` draws the "no change" diagonal for reference;
* a **second point layer with its own data** can highlight the iron genes. Each geom can take a `data =` argument; when it does, that layer ignores the data given to `ggplot()`.

In [ ]:
iron_root <- root_means %>%
  filter(AGI %in% iron_genes) %>%
  mutate(gene = names(iron_genes)[match(AGI, iron_genes)])

ggplot(root_means, aes(x = log_col0, y = log_opt3)) +
  geom_point(alpha = 0.2, size = 0.6, colour = "grey40") +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
  geom_point(data = iron_root, colour = "firebrick", size = 2.5) +
  geom_text(data = iron_root, aes(label = gene), colour = "firebrick",
            hjust = -0.2, vjust = 0.5, size = 3.5)

The iron-uptake genes (IRT1, FRO2, FIT, bHLH38, bHLH39) sit well above the diagonal: *opt3-2* roots switch on the iron-deficiency response although the plants were grown with enough iron. That is the phenotype described in the paper, recovered with nothing more than a scatter plot. `OPT3` itself sits close to the diagonal: the *opt3-2* allele lowers *OPT3* expression mainly in the shoot, which we will check with facets below.

### Adding a trend with `geom_smooth()`

`geom_smooth()` fits a line or curve through the points and draws it with a shaded confidence band. Here we ask whether the spread between the genotypes depends on how highly a gene is expressed, by plotting the difference (`log_opt3 - log_col0`, a log2 fold change) against average expression. This kind of figure is called an **MA plot**, and you will make proper ones from DESeq2 results in M4L3.

`method = "gam"` fits a smooth curve; `method = "lm"` would fit a straight line. With this many points the fit takes a few seconds.

In [ ]:
root_means <- root_means %>%
  mutate(avg_expr = (log_col0 + log_opt3) / 2,
         log2_fc  = log_opt3 - log_col0)

ggplot(root_means, aes(x = avg_expr, y = log2_fc)) +
  geom_point(alpha = 0.15, size = 0.5) +
  geom_hline(yintercept = 0, colour = "steelblue") +
  geom_smooth(method = "gam", colour = "orange")

**Question:** Where on the x axis are the fold changes most spread out? Why do you think lowly expressed genes show such large positive and negative "fold changes"? (Hint: what is the fold change between 1 read and 4 reads?)

<a id='distributions'></a>
## 4. Looking at distributions

Scatter plots show two continuous variables. When one variable is a category (a genotype, a sample, a treatment) and the other is a number, we usually want to see the **distribution** of the numbers in each category.

### Boxplots and violins of whole libraries

A boxplot summarises a distribution with five numbers: the line in the box is the median, the box spans the middle 50 % of values, the whiskers reach out to the most extreme values within 1.5 box-lengths, and points beyond that are drawn individually. A violin plot draws the full shape of the distribution as a mirrored density curve, which shows features (two peaks, for example) that a box hides.

Before any DE analysis it is worth checking that every library has a similar overall distribution of expression. A library that looks very different from the others in its group may have a technical problem. Here we use the 18 untreated root libraries and keep genes with at least one read.

In [ ]:
root_ctrl <- long %>%
  filter(tissue == "root", treatment == "control", count > 0)

options(repr.plot.width = 10, repr.plot.height = 5)

ggplot(root_ctrl, aes(x = library, y = log2(cpm + 1), fill = genotype)) +
  geom_boxplot(outlier.size = 0.3)

ggplot(root_ctrl, aes(x = library, y = log2(cpm + 1), fill = genotype)) +
  geom_violin()

**Question:** Do any libraries stand out? What does the violin show about the distribution of expression that the boxplot does not?

### One gene, many samples

For a single gene, a boxplot of nine libraries per group shows the group difference and the spread among replicates. With so few points it is honest to also draw the points themselves. `geom_jitter()` draws points with a small random sideways shift so that they do not hide each other; `width` controls how far they move. Setting `outlier.shape = NA` in the boxplot stops outliers being drawn twice.

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 5)

irt1_roots <- iron %>%
  filter(gene == "IRT1", tissue == "root")

ggplot(irt1_roots, aes(x = genotype, y = cpm)) +
  geom_boxplot(outlier.shape = NA) +
  geom_jitter(aes(colour = treatment), width = 0.15, height = 0, size = 2)

The y axis is dominated by the very high *opt3-2* values. Putting the y axis on a log scale with `scale_y_log10()` spreads the low values out. Notice that the data are not changed; only the axis is.

In [ ]:
ggplot(irt1_roots, aes(x = genotype, y = cpm)) +
  geom_boxplot(outlier.shape = NA) +
  geom_jitter(aes(colour = treatment), width = 0.15, height = 0, size = 2) +
  scale_y_log10()

<a id='facets'></a>
## 5. Small multiples with facets

Often we want the same plot repeated for several subsets: one panel per gene, or one per tissue. **Facets** do this without a loop.

* `facet_wrap(~ gene)` makes one panel per gene and wraps the panels into a grid.
* `facet_grid(rows ~ cols)` uses one variable for rows and another for columns.

By default all panels share the same axes. Genes are expressed at very different levels, so `scales = "free_y"` gives each panel its own y axis.

The plot below uses root libraries for all eight iron genes. Mapping `fill = treatment` splits each genotype into a control box and a Cd box.

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 6)

iron %>%
  filter(tissue == "root") %>%
  ggplot(aes(x = genotype, y = cpm, fill = treatment)) +
  geom_boxplot() +
  facet_wrap(~ gene, nrow = 2, scales = "free_y")

Note the pipe: we filtered the data and piped it straight into `ggplot()`, so the filtered table is the data for the plot. After `ggplot()` the pieces are joined with `+`, not `%>%`. Mixing them up gives an error.

Now a `facet_grid()` for one gene. Rows are tissues and columns are treatments, so each panel compares the two genotypes under one condition. Here we look at `OPT3` itself.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)

iron %>%
  filter(gene == "OPT3") %>%
  ggplot(aes(x = genotype, y = cpm, colour = genotype)) +
  geom_jitter(width = 0.15, height = 0, size = 2) +
  facet_grid(tissue ~ treatment, scales = "free_y")

**Question:** In which tissue is *OPT3* expressed most strongly in Col-0, and in which tissue does the *opt3-2* mutation reduce it most? How does cadmium change the picture?

<a id='polish'></a>
## 6. Making a plot readable

A plot for a report or a poster needs more than correct marks. Readers should understand it without reading your code. Four kinds of layer handle this:

| Layer | What it controls | Example |
|---|---|---|
| `labs()` | title, subtitle, axis titles, legend titles, caption | `labs(x = "Genotype", fill = "Treatment")` |
| `scale_*()` | how a mapped variable is shown: axis breaks and transformation, colour choices | `scale_y_log10()`, `scale_fill_manual(values = ...)` |
| `theme_*()` | complete preset looks | `theme_bw()`, `theme_classic()`, `theme_minimal()` |
| `theme()` | individual details | `theme(legend.position = "bottom")` |

We take the faceted iron-gene plot and finish it. The plot is first stored in a variable, `p`. Nothing is drawn until the variable is printed, and we can keep adding layers to it.

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 6.5)

p <- iron %>%
  filter(tissue == "root") %>%
  ggplot(aes(x = genotype, y = cpm, fill = treatment)) +
  geom_boxplot(outlier.shape = NA) +
  geom_point(position = position_jitterdodge(jitter.width = 0.1), size = 0.8) +
  facet_wrap(~ gene, nrow = 2, scales = "free_y")

p_final <- p +
  scale_fill_manual(values = c(control = "#9ecae1", Cd = "#fdae6b"),
                    labels = c(control = "Fe-replete control", Cd = "Cadmium")) +
  labs(
    title    = "Iron-homeostasis genes in Arabidopsis roots (GSE79275)",
    subtitle = "Each point is one sequencing library (3 biological x 3 technical replicates per box)",
    x        = NULL,                      # the genotype names are self-explanatory
    y        = "Counts per million",
    fill     = "Growth condition",
    caption  = "Data: Khan et al. 2018, Plant Cell Environ., doi:10.1111/pce.13192"
  ) +
  theme_bw(base_size = 13) +
  theme(legend.position = "bottom",
        strip.text = element_text(face = "italic"))

p_final

A few details worth noticing:

* `position_jitterdodge()` shifts the points sideways *and* lines them up with the dodged control/Cd boxes.
* `scale_fill_manual()` sets both the colours and the legend text. The names in `values` and `labels` must match the values in the data (`control`, `Cd`).
* `x = NULL` removes an axis title that adds nothing.
* `theme_bw()` replaces the grey background; `base_size` scales all text at once.
* The caption gives the data source. Make this a habit.

### Saving a figure

`ggsave()` writes the most recent plot (or the plot you give it) to a file. The file type is taken from the extension: `.png` and `.tiff` for images, `.pdf` or `.svg` for vector graphics that stay sharp at any size. `width` and `height` are in inches by default. The file is written next to this notebook.

In [ ]:
ggsave("iron_genes_roots.png", plot = p_final, width = 11, height = 6.5, dpi = 300)

<a id='busy'></a>
## 7. When more is less

The plot below was written by an AI assistant when it was asked to "show everything interesting about the iron genes in one figure". Every line of code is valid and well commented, and the result is a figure that is hard to read. Generative AI is good at producing code that runs; it is your job to decide whether the figure tells a clear story. Run the cell, then read the code layer by layer and ask what each layer adds.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 8)

iron %>%
  ggplot(aes(x = group, y = cpm, colour = gene, shape = bio_rep)) +
  # every library of every gene in every group, as large semi-transparent points
  geom_jitter(aes(size = count), alpha = 0.6, width = 0.3, height = 0) +
  # a trend line per gene across the eight groups
  geom_smooth(aes(group = gene), method = "loess", se = FALSE, linewidth = 0.8) +
  # a log scale with custom breaks
  scale_y_log10(breaks = c(0.1, 1, 10, 100, 1000, 10000)) +
  scale_colour_brewer(palette = "Set1") +
  labs(
    title    = "Expression of iron homeostasis genes across all genotypes, tissues and treatments",
    subtitle = "Point size = raw count; shape = biological replicate; lines = LOESS trend per gene",
    x = "Genotype / tissue / treatment", y = "CPM (log10 scale)",
    colour = "Gene", shape = "Bio rep", size = "Raw count"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "right",
        plot.title = element_text(face = "bold", size = 14))

Some problems to look for:

* **Eight genes share one panel.** Their colours overlap, and the reader has to match eight colours to a legend.
* **Four aesthetics at once** (colour, shape, size, position). Each one asks the reader to decode another legend, and two of them (`size = count`, `shape = bio_rep`) carry information nobody asked for.
* **A trend line across categories.** The eight groups on the x axis are not an ordered scale, so a smooth curve through them suggests a trend that does not exist. Without points to fit, `loess` may also print warnings.
* **Mixed tissues on one axis.** Root and shoot values are side by side although they answer different questions.

Compare it with `p_final` above, which shows fewer things with more care. You will be asked to improve this figure in Exercise 4.

<a id='exercises'></a>
## 8. Exercises

Write your code in the empty cells. Each exercise builds on objects created earlier in this notebook (`long`, `iron`, `root_means`, `lib_sizes`), so run the cells above first.

**Exercise 1.** Make a violin plot of `FRO2` expression in **root** libraries, with `group` on the x axis and `cpm` on the y axis. Add the individual libraries on top with `geom_jitter()`. Is a log scale on the y axis helpful here?

In [ ]:
# Exercise 1: your code here

**Exercise 2.** Make a new grouping column and plot it. Use `mutate()` with `ifelse()` and `grepl()` to add a column called `fit_target` to `iron`, set to `"FIT target"` for IRT1 and FRO2 and to `"other"` for every other gene. (Hint: `grepl("IRT1|FRO2", gene)`; the `|` means "or".) Then draw a boxplot of `log2(cpm + 1)` with `fit_target` on the x axis and `genotype` as the fill, using only root libraries.

In [ ]:
# Exercise 2: your code here

**Exercise 3.** Take your boxplot from Exercise 2 and make it ready for a report: add a title, clear axis and legend titles with `labs()`, choose your own fill colours with `scale_fill_manual()`, and apply a complete theme. Then save it with `ggsave()` as a PDF.

In [ ]:
# Exercise 3: your code here

**Exercise 4.** Redesign the "When more is less" figure. Decide what one or two messages the figure should carry (for example: "the iron-uptake genes are much higher in *opt3-2* roots, and Cd weakens this"), then remove or change layers until the figure says that and nothing else. Facets are probably your friend. In a markdown cell, list the changes you made and why.

In [ ]:
# Exercise 4: your code here

**Exercise 5.** Using `lib_sizes`, make a scatter plot of `total_reads` against `genes_detected`, with `colour = tissue` and `shape = treatment`, and add `geom_smooth(method = "lm")`. Then put `colour = tissue` *inside* `geom_smooth()` instead of in `ggplot()` and compare. What changes, and why?

In [ ]:
# Exercise 5: your code here

<a id='ahead'></a>
## 9. Looking ahead: a heat map of DE genes

The rest of the module moves from raw counts to proper differential expression. As a preview, the cell below runs a minimal DESeq2 analysis of the untreated root libraries (*opt3-2* vs Col-0) and draws the 40 most significant genes as a heat map with `pheatmap`. Do not worry about the details: M4L3 walks through these plots and M4P1 explains the statistics. Watch for two things:

* the technical replicates are first **summed** per biological replicate, so the analysis has 3 vs 3 samples, not 9 vs 9;
* the heat map shows *scaled* expression: each row is centred to mean 0 and divided by its standard deviation, so colours compare samples within a gene, not genes with each other.

Red text printed by DESeq2 is progress information, not an error.

In [ ]:
library(DESeq2)
library(pheatmap)

# untreated root libraries only
root_cols <- grep("^[CO]R[0-9]", colnames(counts), value = TRUE)
mat <- as.matrix(counts[, root_cols])
rownames(mat) <- counts$AGI

# sum technical replicates: CR1_5, CR1_6, CR1_7 -> CR1
bio_id <- sub("_[0-9]+\\.bam$", "", root_cols)
mat_bio <- t(rowsum(t(mat), group = bio_id))

sample_info <- data.frame(
  row.names = colnames(mat_bio),
  genotype  = factor(ifelse(grepl("^O", colnames(mat_bio)), "opt3", "col0"), levels = c("col0", "opt3"))
)

dds <- DESeqDataSetFromMatrix(countData = mat_bio, colData = sample_info, design = ~ genotype)
dds <- dds[rowSums(counts(dds)) >= 10, ]
dds <- DESeq(dds)
res <- results(dds, contrast = c("genotype", "opt3", "col0"))
summary(res, alpha = 0.05)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 9)

top40 <- head(order(res$padj), 40)
vsd <- vst(dds, blind = FALSE)

# label rows with the short names where we have one
row_labels <- rownames(vsd)[top40]
known <- match(row_labels, iron_genes)
row_labels[!is.na(known)] <- names(iron_genes)[known[!is.na(known)]]

pheatmap(assay(vsd)[top40, ],
         scale = "row",
         annotation_col = sample_info,
         labels_row = row_labels,
         main = "Top 40 DE genes, opt3-2 vs Col-0 roots (GSE79275)")

**Question:** Do the samples cluster by genotype? Which of the iron genes from this lab appear among the top 40?

## Further reading

* Wickham H, Grolemund G. *R for Data Science*, 1st ed. O'Reilly; 2017. Chapter 3, "Data visualisation": <https://r4ds.had.co.nz/data-visualisation.html>. A longer tutorial on the grammar of graphics.
* Wickham H. *ggplot2: Elegant Graphics for Data Analysis*. 2nd ed. Springer; 2016. [doi:10.1007/978-3-319-24277-4](https://doi.org/10.1007/978-3-319-24277-4).
* The `ggplot2` reference pages list every geom, scale and theme with examples: <https://ggplot2.tidyverse.org/reference/>.
* Data: Khan MA, et al. *Plant Cell Environ.* 2018;41(10):2263-2276. [doi:10.1111/pce.13192](https://doi.org/10.1111/pce.13192). See [`data/README.md`](../../../data/README.md).